# 02 — Inspect and validate the data before fitting

**Plain-language question:** Is the data trustworthy enough to train on?

**Why this matters:** an algorithm cannot repair a missing column, duplicated
identity, impossible label, or unexplained change in the dataset.

**Estimated time:** 45–55 minutes.
**Prerequisite:** lessons 00–01; you know rows, features, labels, and prediction
time.


## Preflight

Run the environment check before touching the data.


In [ ]:
import importlib.util
import sys

required = ("mlflow", "pandas", "sklearn", "aai_local_classification")
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(
        "This notebook is using the wrong Python kernel. Close this Jupyter "
        "server, run `make notebook` from examples/local-classification, or "
        "select the 'AAI Local Classification' kernel. Missing: " + ", ".join(missing)
    )

import pandas as pd

from aai_local_classification.learning import study_root
from aai_local_classification.settings import load_settings
from aai_local_classification.tracking import local_paths

settings = load_settings()
root = study_root()
paths = local_paths(root)
print(f"✓ Python {sys.version_info.major}.{sys.version_info.minor}: {sys.executable}")
print("✓ Course imports are available")
print(f"✓ Learner state: {root}")


In [ ]:
from aai_local_classification.contracts import SplitName
from aai_local_classification.data import (
    load_split,
    validate_dataset,
)
from aai_local_classification.workflow import ensure_prepared

manifest = ensure_prepared(settings, root)
train = load_split(settings, SplitName.TRAIN, paths.data_root)
validation = load_split(settings, SplitName.VALIDATION, paths.data_root)
print(f"✓ Train {train.shape}; validation {validation.shape}")


### What you should see

Train `(2160, 12)` and validation `(360, 12)`. We do not load test data in this
lesson. The later final exam remains sealed.

### Words introduced

| Word | Plain meaning | Example check |
|---|---|---|
| schema | Expected column names and data types | `monthly_fee` is numeric |
| missing value | An unknown entry, represented as `NaN`/`<NA>` | missing usage |
| duplicate | A row identity appearing more than once | repeated `account_id` |


## Start with shape, sample rows, and data types


In [ ]:
train.head(3)


In [ ]:
schema_view = pd.DataFrame(
    {
        "dtype": train.dtypes.astype(str),
        "missing_rows": train.isna().sum(),
        "distinct_values": train.nunique(dropna=True),
    }
)
schema_view


### How to interpret this

The table is a first inspection, not proof of quality. Numeric features should
load as numbers, the snapshot should parse as a date, and only the two columns
designed with missingness should have missing rows. A type can be technically
valid yet semantically wrong, so meanings still require a data contract.


## Make important checks visible

**Before you run this:** predict the correct duplicate count and allowed label
set for this course.


In [ ]:
visible_checks = pd.Series(
    {
        "duplicate_account_ids": int(train.account_id.duplicated().sum()),
        "missing_account_ids": int(train.account_id.isna().sum()),
        "invalid_snapshot_dates": int(
            pd.to_datetime(train.snapshot_date, errors="coerce").isna().sum()
        ),
        "label_values": str(sorted(train.churned_30d.unique())),
        "maximum_feature_missing_rate": train[list(settings.features.model_columns)]
        .isna()
        .mean()
        .max(),
    },
    name="observed",
)
visible_checks.to_frame()


### What you should see

Zero duplicate/missing IDs, zero invalid dates, labels `[0, 1]`, and maximum
feature missingness around 3%. The course contract permits at most 10%.

These checks fail fast because training on structurally unexpected data would
create misleading evidence.


## Look for change across time

**Prevalence** is the share of rows whose label is positive. It can change even
when the schema stays identical. We can inspect training and validation because
both may guide development; test stays unopened.


In [ ]:
cohort_summary = pd.DataFrame(
    {
        "rows": [len(train), len(validation)],
        "positive_rate": [train.churned_30d.mean(), validation.churned_30d.mean()],
        "usage_missing_rate": [
            train.usage_hours_30d.isna().mean(),
            validation.usage_hours_30d.isna().mean(),
        ],
        "channel_missing_rate": [
            train.signup_channel.isna().mean(),
            validation.signup_channel.isna().mean(),
        ],
    },
    index=["train", "validation"],
)
cohort_summary


### What you should see

Training prevalence is about 15.1%; validation is about 19.4%. A later cohort
already looks different. That observation motivates a time-based split; it does
not by itself explain why churn changed.


In [ ]:
import matplotlib.pyplot as plt

monthly = train.assign(month=train.snapshot_date.dt.to_period("M").astype(str))
monthly_rate = monthly.groupby("month").churned_30d.mean()
ax = monthly_rate.plot(figsize=(9, 3), marker="o", title="Training churn rate by month")
ax.set_ylabel("share with churned_30d = 1")
ax.set_xlabel("snapshot month")
plt.xticks(rotation=45)
plt.tight_layout()


The plot is **exploratory data analysis (EDA)**: a way to notice patterns worth
investigating. Association is not causation. A higher monthly rate does not tell
us which feature caused it or whether it will continue.


## Version the exact dataset

### Words introduced

| Word | Plain meaning | Here |
|---|---|---|
| manifest | A small inventory describing data files | `manifest.json` |
| digest | A fingerprint that changes when bytes change | SHA-256 text |
| lineage | Evidence of where an input came from | generator + config + split |

A digest detects change; it does not contain the data, prove correctness, or
make the data representative.


In [ ]:
manifest_table = pd.DataFrame(
    [
        {
            "split": item.split.value,
            "rows": item.row_count,
            "dates": f"{item.start_date} to {item.end_date}",
            "positive_rate": item.positive_rate,
            "sha256": item.sha256[:12] + "…",
        }
        for item in manifest.artifacts
    ]
)
manifest_table


Notice that the test row count and dates are visible, but its `positive_rate` is
blank. The manifest lets us verify the sealed file without exposing its label
summary during development.


In [ ]:
import hashlib

first = hashlib.sha256(b"same text").hexdigest()[:12]
second = hashlib.sha256(b"same text!").hexdigest()[:12]
pd.Series({"same text": first, "one-character change": second})


### What you should see

The two short fingerprints differ completely. The course stores full 64-character
digests; we shorten them only for display.

Now compare our visible checks with the packaged validator used by repeatable
jobs.


In [ ]:
packaged_quality = validate_dataset(train, settings)
pd.Series(packaged_quality, name="observed").to_frame()


The helper repeats enforceable schema and quality rules. We inspected the
important operations first, so the function is now a reusable safety boundary
rather than a black box.

### Misconception check

Passing these checks means “matches this authored contract.” It does not mean
the synthetic sample represents real customers, is fair, or supports a useful
business decision.


### Guided exercise

Duplicate one row in a scratch DataFrame. Do not alter `train`. Predict which
quality rule should reject the scratch copy.


In [ ]:
exercise_with_duplicate = pd.concat([train, train.iloc[[0]]], ignore_index=True)
exercise_duplicate_count = int(exercise_with_duplicate.account_id.duplicated().sum())
print(f"Duplicate IDs in scratch data: {exercise_duplicate_count}")


**Self-check:** the scratch copy should contain exactly one duplicated ID. The
solution catches the expected validation error so Restart-and-Run-All still
finishes successfully.

<details><summary>Solution explanation</summary>

The validator rejects duplicated account identities before any model fit. The
original `train` object and persisted data remain unchanged.
</details>


In [ ]:
# Reference solution — run after your attempt
assert exercise_duplicate_count == 1
try:
    validate_dataset(exercise_with_duplicate, settings)
except ValueError as error:
    print(f"✓ Expected rejection: {error}")
else:
    raise AssertionError("The duplicate should have been rejected")


## MLOps bridge

Later MLflow runs log the dataset input and manifest digest. On Databricks, the
equivalent source would normally be a versioned Unity Catalog Delta table rather
than local CSV files.

## Recap

- Inspect shape, rows, types, missingness, identities, labels, and time cohorts
  before fitting.
- A manifest and digest make change detectable and traceable.
- Quality rules are necessary boundaries, not proof of real-world usefulness.

**Evidence created:** the same manifest and split files from lesson 00; no model
has been trained. Reruns verify rather than silently replace them.

**Ready for 03?** You can explain why validation may be inspected while the
frozen test label summary remains sealed.
